In [13]:
# Import Library

import mne
import numpy as np
from scipy.stats import skew, kurtosis
from mne.time_frequency import psd_array_welch
import gc
import psutil
import matplotlib.pyplot as plt

def show_ram():
    ram = psutil.virtual_memory()
    print(f"Available RAM: {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB")

show_ram()

Available RAM: 6.4 GB / 8.2 GB


In [2]:
# Import raw data (.fif)

raw = mne.io.read_raw_fif(
    "sub10_sess1_50_ica_eeg.fif",
    preload=True
)

print(raw)

Opening raw data file sub10_sess1_50_ica_eeg.fif...
    Range : 0 ... 4326399 =      0.000 ...  4224.999 secs
Ready.
Opening raw data file /home/cindy/BHSFinal/sub10_sess1_50_ica_eeg-1.fif...
    Range : 4326400 ... 4958827 =   4225.000 ...  4842.604 secs
Ready.
Reading 0 ... 4958827  =      0.000 ...  4842.604 secs...
<Raw | sub10_sess1_50_ica_eeg.fif, 124 x 4958828 (4842.6 s), ~4.58 GiB, data loaded>


In [3]:
# check events and event id
events, event_id = mne.events_from_annotations(raw)

Used Annotations descriptions: [np.str_('0, Imagination_a_flower_high_10###my_stream_name'), np.str_('0, Imagination_a_flower_high_11###my_stream_name'), np.str_('0, Imagination_a_flower_high_17###my_stream_name'), np.str_('0, Imagination_a_flower_high_23###my_stream_name'), np.str_('0, Imagination_a_flower_high_4###my_stream_name'), np.str_('0, Imagination_a_flower_high_5###my_stream_name'), np.str_('0, Imagination_a_flower_low_10###my_stream_name'), np.str_('0, Imagination_a_flower_low_13###my_stream_name'), np.str_('0, Imagination_a_flower_low_17###my_stream_name'), np.str_('0, Imagination_a_flower_low_23###my_stream_name'), np.str_('0, Imagination_a_flower_low_25###my_stream_name'), np.str_('0, Imagination_a_flower_low_5###my_stream_name'), np.str_('0, Imagination_a_flower_normal_10###my_stream_name'), np.str_('0, Imagination_a_flower_normal_11###my_stream_name'), np.str_('0, Imagination_a_flower_normal_12###my_stream_name'), np.str_('0, Imagination_a_flower_normal_16###my_stream_n

In [4]:
# clean the name so that it only countain "perception_modalities_unique code"

clean_event_id = {}

for k, v in event_id.items():
    clean_name = k.split(", ")[-1]      # remove "0, "
    clean_name = clean_name.replace("###my_stream_name", "")
    clean_event_id[clean_name] = v

# print(clean_event_id)

In [5]:
# Create modalities group

# Image
image_events = [
    k for k in clean_event_id.keys()
    if k.startswith("Perception_image")
]


# Text
text_events = [
    k for k in clean_event_id.keys()
    if k.startswith("Perception_t")
]


# Audio
audio_events = [
    k for k in clean_event_id.keys()
    if k.startswith("Perceptiona")
]

print("Image:", len(image_events))
print("Text:", len(text_events))
print("Audio:", len(audio_events))

print("Image:", image_events)
print("Text:", text_events)
print("Audio:", audio_events)

Image: 9
Text: 15
Audio: 75
Image: ['Perception_image_flower_c', 'Perception_image_flower_m', 'Perception_image_flower_s', 'Perception_image_guitar_c', 'Perception_image_guitar_m', 'Perception_image_guitar_s', 'Perception_image_penguin_c', 'Perception_image_penguin_m', 'Perception_image_penguin_s']
Text: ['Perception_t_flower_1', 'Perception_t_flower_2', 'Perception_t_flower_3', 'Perception_t_flower_4', 'Perception_t_flower_5', 'Perception_t_guitar_1', 'Perception_t_guitar_2', 'Perception_t_guitar_3', 'Perception_t_guitar_4', 'Perception_t_guitar_5', 'Perception_t_penguin_1', 'Perception_t_penguin_2', 'Perception_t_penguin_3', 'Perception_t_penguin_4', 'Perception_t_penguin_5']
Audio: ['Perceptiona_flower_high_10', 'Perceptiona_flower_high_11', 'Perceptiona_flower_high_17', 'Perceptiona_flower_high_23', 'Perceptiona_flower_high_4', 'Perceptiona_flower_high_5', 'Perceptiona_flower_low_10', 'Perceptiona_flower_low_13', 'Perceptiona_flower_low_17', 'Perceptiona_flower_low_23', 'Perception

In [6]:
# Convert the new groups into event_id

image_event_id = {k: clean_event_id[k] for k in image_events}
text_event_id  = {k: clean_event_id[k] for k in text_events}
audio_event_id = {k: clean_event_id[k] for k in audio_events}

print("Image: ", image_event_id)
print("Text: ", text_event_id)
print("Audio: ", audio_event_id)

Image:  {'Perception_image_flower_c': 101, 'Perception_image_flower_m': 102, 'Perception_image_flower_s': 103, 'Perception_image_guitar_c': 104, 'Perception_image_guitar_m': 105, 'Perception_image_guitar_s': 106, 'Perception_image_penguin_c': 107, 'Perception_image_penguin_m': 108, 'Perception_image_penguin_s': 109}
Text:  {'Perception_t_flower_1': 110, 'Perception_t_flower_2': 111, 'Perception_t_flower_3': 112, 'Perception_t_flower_4': 113, 'Perception_t_flower_5': 114, 'Perception_t_guitar_1': 115, 'Perception_t_guitar_2': 116, 'Perception_t_guitar_3': 117, 'Perception_t_guitar_4': 118, 'Perception_t_guitar_5': 119, 'Perception_t_penguin_1': 120, 'Perception_t_penguin_2': 121, 'Perception_t_penguin_3': 122, 'Perception_t_penguin_4': 123, 'Perception_t_penguin_5': 124}
Audio:  {'Perceptiona_flower_high_10': 125, 'Perceptiona_flower_high_11': 126, 'Perceptiona_flower_high_17': 127, 'Perceptiona_flower_high_23': 128, 'Perceptiona_flower_high_4': 129, 'Perceptiona_flower_high_5': 130, 'P

In [7]:
# Combine all event_id
all_event_id = {}

all_event_id.update(image_event_id)
all_event_id.update(text_event_id)
all_event_id.update(audio_event_id)

# print(all_event_id)

In [8]:
# Create epoch 

epochs = mne.Epochs(
    raw,
    events,
    event_id=all_event_id,
    tmin=0,
    tmax=2.0,
    baseline=None,
    preload=True
)

# Free raw
del raw
gc.collect()
print("raw deleted")
show_ram()

Not setting metadata
450 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 450 events and 2049 original time points ...
0 bad epochs dropped
raw deleted
Available RAM: 6.4 GB / 8.2 GB


In [9]:
# split epochs by modality

epochs_text = epochs[text_events]
epochs_image = epochs[image_events]
epochs_audio = epochs[audio_events]
show_ram()

Available RAM: 5.5 GB / 8.2 GB


In [10]:
# Free raw

del epochs
gc.collect()
show_ram()

Available RAM: 6.4 GB / 8.2 GB


In [ ]:
#######################################
# ERP Analysis
#######################################

In [15]:
# Compute evoked (averaged ERP) per condition
evoked_image_av = epochs_image.average()
evoked_text_av  = epochs_text.average()
evoked_audio_av = epochs_audio.average()

# ── Butterfly plot (all channels) per condition ──────────
conditions = [
    ('image', evoked_image_av, 'cornflowerblue'),
    ('text',  evoked_text_av,  'tomato'),
    ('audio', evoked_audio_av, 'mediumseagreen')
]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, (name, evoked, color) in zip(axes, conditions):
    evoked.plot(axes=ax, spatial_colors=False, show=False, time_unit='ms')
    ax.set_title(f"ERP — {name.upper()} stimulus", color=color, fontweight='bold')
    ax.axvline(0, color='black', lw=1.5, ls='--', label='Stimulus onset')
fig.suptitle("ERP Butterfly Plots by Condition", fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig("eeg_plots/11_erp_butterfly.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("→ Saved: eeg_plots/11_erp_butterfly.png")

# ── Single-channel comparison (Pz — best for P300) ──────
fig, ax = plt.subplots(figsize=(12, 5))
ch = 'Pz'
if ch in evoked_image_av.ch_names:
    ch_idx = evoked_image_av.ch_names.index(ch)
    times_ms = evoked_image_av.times * 1000
    for name, evoked, color in conditions:
        ax.plot(times_ms, evoked.data[ch_idx] * 1e6, label=name, color=color, lw=2)
    ax.axvline(0, color='black', lw=1.5, ls='--', alpha=0.7)
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude (µV)")
    ax.set_title(f"ERP at {ch}: Image vs Text vs Audio", fontsize=13, fontweight='bold')
    ax.legend(fontsize=12)
    ax.set_xlim([-200, 1000])
    fig.tight_layout()
    fig.savefig("eeg_plots/12_erp_Pz_comparison.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    print("→ Saved: eeg_plots/12_erp_Pz_comparison.png")

# ── Topomap comparison at key timepoints ─────────────────
"""
Topomaps show WHERE on the scalp the signal difference appears.
P300 at ~300ms: central/parietal positivity — larger = more cognitive load
"""
timepoints = [0.1, 0.2, 0.3, 0.5]   # seconds post-stimulus

fig, axes = plt.subplots(3, len(timepoints), figsize=(14, 8))
for row, (name, evoked, color) in enumerate(conditions):
    for col, tp in enumerate(timepoints):
        mne.viz.plot_topomap(
            evoked.copy().crop(tmin=tp-0.01, tmax=tp+0.01).data.mean(axis=1),
            evoked.info,
            axes=axes[row, col],
            show=False,
            cmap='RdBu_r'
        )
        if row == 0:
            axes[row, col].set_title(f"{int(tp*1000)} ms", fontsize=11, fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(name, fontsize=11, color=color, fontweight='bold')
fig.suptitle("Brain Topographies at Key Timepoints\n(Image / Text / Audio)", 
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig("eeg_plots/13_topomaps_by_condition.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("→ Saved: eeg_plots/13_topomaps_by_condition.png")

# ── Joint plot per condition (topo + waveform) ───────────
for name, evoked, color in conditions:
    try:
        fig = evoked.plot_joint(
            times=[0.1, 0.2, 0.3, 0.5],
            title=f"ERP Joint Plot — {name.upper()}",
            show=False
        )
        fig.savefig(f"eeg_plots/14_joint_{name}.png", dpi=100, bbox_inches='tight')
        plt.close(fig)
        print(f"→ Saved: eeg_plots/14_joint_{name}.png")
    except Exception as e:
        print(f"  Joint plot skipped for {name}: {e}")

gc.collect()

# Different waves
diff_image_audio = mne.combine_evoked([evoked_image_av, evoked_audio_av], weights=[1, -1])
diff_text_image  = mne.combine_evoked([evoked_text_av,  evoked_image_av], weights=[1, -1])

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, (diff, label) in zip(axes, [
    (diff_image_audio, 'Image − Audio'),
    (diff_text_image,  'Text − Image')
]):
    diff.plot(axes=ax, spatial_colors=True, show=False, time_unit='ms')
    ax.set_title(f"Difference Wave: {label}", fontweight='bold')
    ax.axvline(0, color='black', lw=1.5, ls='--')

fig.suptitle("Difference Waves: Stimulus Condition Contrasts", 
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig("eeg_plots/15_difference_waves.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("→ Saved: eeg_plots/15_difference_waves.png")

gc.collect()


→ Saved: eeg_plots/11_erp_butterfly.png
→ Saved: eeg_plots/12_erp_Pz_comparison.png
→ Saved: eeg_plots/13_topomaps_by_condition.png
No projector specified for this dataset. Please consider the method self.add_proj.
→ Saved: eeg_plots/14_joint_image.png
No projector specified for this dataset. Please consider the method self.add_proj.
→ Saved: eeg_plots/14_joint_text.png
No projector specified for this dataset. Please consider the method self.add_proj.
→ Saved: eeg_plots/14_joint_audio.png


/tmp/ipykernel_519/1064027595.py:103: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


→ Saved: eeg_plots/15_difference_waves.png


1458

In [ ]:
#######################################
# Time frequency analysis
#######################################

In [17]:
freqs = np.arange(4, 40, 1)
n_cycles = freqs / 2.0

tfr_image = mne.time_frequency.tfr_morlet(
    epochs_image, freqs=freqs, n_cycles=n_cycles,
    return_itc=False, average=True, n_jobs=1
)
tfr_text = mne.time_frequency.tfr_morlet(
    epochs_text, freqs=freqs, n_cycles=n_cycles,
    return_itc=False, average=True, n_jobs=1
)
tfr_audio = mne.time_frequency.tfr_morlet(
    epochs_audio, freqs=freqs, n_cycles=n_cycles,
    return_itc=False, average=True, n_jobs=1
)

tfr_conditions = [
    ('image', tfr_image),
    ('text', tfr_text),
    ('audio', tfr_audio)
]

ch_tfr = 'Pz'
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (name, tfr) in zip(axes, tfr_conditions):
    tfr.plot(
        picks=[ch_tfr], axes=ax, show=False, colorbar=(name == 'audio'),
        baseline=(-0.2, 0), mode='logratio',
        title=f"TFR ({name}) at {ch_tfr}"
    )
fig.suptitle(
    f"Time-Frequency Power at {ch_tfr}: Image vs Text vs Audio",
    fontsize=13, fontweight='bold'
)
fig.tight_layout()
fig.savefig("eeg_plots/16_tfr_comparison.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("→ Saved: eeg_plots/16_tfr_comparison.png")

gc.collect()


NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
→ Saved: eeg_plots/16_tfr_comparison.png


164630